# Explore Pipeline Outputs

Quick inspection of generated triplets, word counts, and fact-check results.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from utils import read_jsonl

## Load Triplets

In [ ]:
triplets_path = PROJECT_ROOT / "data" / "triplets" / "fact_checked_triplets.jsonl"
if triplets_path.exists():
    triplets = read_jsonl(triplets_path)
    print(f"Loaded {len(triplets)} triplets")
else:
    print(f"File not found: {triplets_path}")
    print("Run the pipeline first.")
    triplets = []

## Sample Triplets

In [ ]:
for i, t in enumerate(triplets[:3]):
    print(f"\n{'='*80}")
    print(f"Triplet {i+1}: {t.get('item_id', 'N/A')}")
    print(f"Task: {t.get('legalbench_task', 'N/A')} | Domain: {t.get('domain', 'N/A')}")
    print(f"Ground truth: {t.get('ground_truth', 'N/A')}")
    print(f"Fact-check pass: {t.get('fact_check_pass', 'N/A')}")
    print(f"\n--- Expert ---\n{t.get('expert_prompt', '')[:300]}")
    print(f"\n--- Naive-Calm ---\n{t.get('naive_calm_prompt', '')[:300]}")
    print(f"\n--- Naive-Distressed ---\n{t.get('naive_distressed_prompt', '')[:300]}")

## Word Count Distribution

In [ ]:
if triplets:
    wc_expert = [t.get("word_count_expert", 0) for t in triplets]
    wc_calm = [t.get("word_count_naive_calm", 0) for t in triplets]
    wc_distressed = [t.get("word_count_naive_distressed", 0) for t in triplets]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(wc_expert, bins=20, color="#2196F3", edgecolor="white")
    axes[0].set_title("Expert Word Count")
    axes[1].hist(wc_calm, bins=20, color="#4CAF50", edgecolor="white")
    axes[1].set_title("Naive-Calm Word Count")
    axes[2].hist(wc_distressed, bins=20, color="#FF9800", edgecolor="white")
    axes[2].set_title("Naive-Distressed Word Count")
    plt.tight_layout()
    plt.show()

    print(f"Expert:     mean={sum(wc_expert)/len(wc_expert):.0f}")
    print(f"Calm:       mean={sum(wc_calm)/len(wc_calm):.0f}")
    print(f"Distressed: mean={sum(wc_distressed)/len(wc_distressed):.0f}")

## Fact-Check Pass Rate

In [ ]:
if triplets:
    passed = sum(1 for t in triplets if t.get("fact_check_pass"))
    failed = len(triplets) - passed
    print(f"Passed: {passed}/{len(triplets)} ({passed/len(triplets):.1%})")
    print(f"Failed: {failed}/{len(triplets)} ({failed/len(triplets):.1%})")

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie([passed, failed], labels=["Passed", "Failed"],
           colors=["#4CAF50", "#F44336"], autopct="%1.1f%%", startangle=90)
    ax.set_title("Fact-Check Results")
    plt.show()

## Scored Results

In [ ]:
scored_path = PROJECT_ROOT / "data" / "eval" / "scored_outputs.csv"
if scored_path.exists():
    scored = pd.read_csv(scored_path)
    print(f"Loaded {len(scored)} scored rows")
    display(scored.groupby("condition")["correct"].agg(["mean", "sum", "count"]))
else:
    print(f"File not found: {scored_path}")
    print("Run scripts 07 and 08 first.")